# Notebook 01 - Data Generation

**Project:** What Does Poor Service Cost a Business? - Customer Operations Intelligence

**Objective:** Load and profile the UCI Online Retail II dataset (the real transactions foundation), generated three synthetic tables - Customers, Service Tickets, and Operations that link to it. All four raw tables are saved with deliberate, realistic messiness ready for cleaning in Notebook 02.

**Inputs:** `online_retail_II.xlsx` (UCI dataset)

**Outputs:**
- `data/raw/transactions_raw.csv`
- `data/raw/customers_raw.csv`
- `data/raw/tickets_raw.csv`
- `data/raw/operations_raw.csv`

---

## 0. Setup — Libraries and Folder Structure

In [1]:
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

# Reproducibility — every run produces the same synthetic data
np.random.seed(42)
random.seed(42)

# Create folder structure
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

print('Libraries loaded.')
print('Folder structure ready.')

Libraries loaded.
Folder structure ready.


---
## 1. Load UCI Online Retail II Dataset

The dataset contains two sheets covering 2009-2011. load both and combine them into a single transactions table. This is the real data foundation where all customer IDs used for synthetic table generation are sourced from here.

In [2]:
print('Loading Year 2009-2010...')
df1 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2009-2010')

print('Loading Year 2010-2011...')
df2 = pd.read_excel('online_retail_II.xlsx', sheet_name='Year 2010-2011')

transactions = pd.concat([df1, df2], ignore_index=True)

print(f'\nTotal rows loaded: {len(transactions):,}')
print(f'Columns: {transactions.columns.tolist()}')

Loading Year 2009-2010...
Loading Year 2010-2011...

Total rows loaded: 1,067,371
Columns: ['Invoice', 'StockCode', 'Description', 'Quantity', 'InvoiceDate', 'Price', 'Customer ID', 'Country']


### 1.1 Initial Data Profile

Before touching anything, raw data is profiled. This is to understand the profile before cleaning.

In [3]:
print('=== TRANSACTIONS RAW PROFILE ===')
print(f'Rows: {len(transactions):,}')
print(f'Columns: {transactions.shape[1]}')
print(f'Date range: {transactions["InvoiceDate"].min()} to {transactions["InvoiceDate"].max()}')
print(f'Unique customers (with ID): {transactions["Customer ID"].nunique():,}')
print(f'Unique countries: {transactions["Country"].nunique()}')
print(f'Cancellation invoices (prefix C): {transactions["Invoice"].astype(str).str.startswith("C").sum():,}')
print()
print('--- Null counts per column ---')
print(transactions.isnull().sum())
print()
print('--- Top 10 countries by transaction volume ---')
print(transactions['Country'].value_counts().head(10))

=== TRANSACTIONS RAW PROFILE ===
Rows: 1,067,371
Columns: 8
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Unique customers (with ID): 5,942
Unique countries: 43
Cancellation invoices (prefix C): 19,494

--- Null counts per column ---
Invoice             0
StockCode           0
Description      4382
Quantity            0
InvoiceDate         0
Price               0
Customer ID    243007
Country             0
dtype: int64

--- Top 10 countries by transaction volume ---
Country
United Kingdom    981330
EIRE               17866
Germany            17624
France             14330
Netherlands         5140
Spain               3811
Switzerland         3189
Belgium             3123
Portugal            2620
Australia           1913
Name: count, dtype: int64


In [4]:
print('--- Price and Quantity ranges ---')
print(transactions[['Quantity', 'Price']].describe().round(2))
print()
print(f'Negative quantities (returns/errors): {(transactions["Quantity"] < 0).sum():,}')
print(f'Zero prices: {(transactions["Price"] == 0).sum():,}')
print(f'Negative prices: {(transactions["Price"] < 0).sum():,}')

--- Price and Quantity ranges ---
         Quantity       Price
count  1067371.00  1067371.00
mean         9.94        4.65
std        172.71      123.55
min     -80995.00   -53594.36
25%          1.00        1.25
50%          3.00        2.10
75%         10.00        4.15
max      80995.00    38970.00

Negative quantities (returns/errors): 22,950
Zero prices: 6,202
Negative prices: 5


### 1.2 Add Revenue Column and Save Transactions Raw

Add a `Revenue` column (Quantity × Price) so it is available for analysis. I intentionally leave all the messiness — nulls, negatives & cancellations. Notebook 02 handles cleaning.

In [5]:
# Add revenue column — will contain negatives from returns/cancellations (intentional)
transactions['Revenue'] = transactions['Quantity'] * transactions['Price']

# Rename columns to snake_case for consistency across all tables
transactions.columns = [
    'invoice', 'stock_code', 'description', 'quantity',
    'invoice_date', 'price', 'customer_id', 'country', 'revenue'
]

# Save raw transactions
transactions.to_csv('data/raw/transactions_raw.csv', index=False)

print(f'transactions_raw.csv saved — {len(transactions):,} rows')
print(transactions.head(3))

transactions_raw.csv saved — 1,067,371 rows
  invoice stock_code                          description  quantity  \
0  489434      85048  15CM CHRISTMAS GLASS BALL 20 LIGHTS        12   
1  489434     79323P                   PINK CHERRY LIGHTS        12   
2  489434     79323W                  WHITE CHERRY LIGHTS        12   

         invoice_date  price  customer_id         country  revenue  
0 2009-12-01 07:45:00   6.95      13085.0  United Kingdom     83.4  
1 2009-12-01 07:45:00   6.75      13085.0  United Kingdom     81.0  
2 2009-12-01 07:45:00   6.75      13085.0  United Kingdom     81.0  


---
## 2. Extract Customer Base

I extract all unique Customer IDs from the real transaction data. These become the spine of the customer table where every synthetic attribute I add is anchored to a real customer who actually transacted in the dataset.

In [6]:
# Extract customers who have an ID (exclude guest checkouts)
known_customers = transactions.dropna(subset=['customer_id'])['customer_id'].unique()
known_customers = [int(c) for c in known_customers]

print(f'Unique customers with ID: {len(known_customers):,}')
print(f'Sample customer IDs: {known_customers[:5]}')

Unique customers with ID: 5,942
Sample customer IDs: [13085, 13078, 15362, 18102, 12682]


---
## 3. Generate Customers Table (Synthetic Enrichment)

For each real customer ID I generate synthetic attributes - segment, CLV tier, acquisition channel, and preferred region. 

**Deliberate messiness introduced:**
- Missing CLV values for ~8% of customers (new customers with insufficient history)
- Inconsistent segment labelling (mixed casing and abbreviations)
- ~5% of customers with no country recorded (data migration gap)
- Duplicate row for ~2% of customers (system sync error)

In [7]:
n = len(known_customers)

# Segments — weighted so retail is most common (realistic)
segment_pool = (
    ['Retail'] * 50 +
    ['retail'] * 5 +          # inconsistent casing (messiness)
    ['RETAIL'] * 3 +          # inconsistent casing (messiness)
    ['Corporate'] * 25 +
    ['corporate'] * 3 +       # inconsistent casing (messiness)
    ['Wholesale'] * 14
)

segments = np.random.choice(segment_pool, size=n)

# CLV — corporate highest, wholesale mid, retail lower on average
def assign_clv(seg):
    seg_clean = seg.strip().lower()
    if seg_clean == 'corporate':
        return round(np.random.uniform(2000, 15000), 2)
    elif seg_clean == 'wholesale':
        return round(np.random.uniform(800, 5000), 2)
    else:
        return round(np.random.uniform(50, 1200), 2)

clv_values = [assign_clv(s) for s in segments]

# Introduce ~8% missing CLV (new customers)
missing_clv_idx = np.random.choice(n, size=int(n * 0.08), replace=False)
clv_array = np.array(clv_values, dtype=object)
clv_array[missing_clv_idx] = np.nan

# Acquisition channels
channels = np.random.choice(
    ['Online', 'Referral', 'Direct Sales', 'Trade Show', 'Partner'],
    size=n,
    p=[0.40, 0.20, 0.20, 0.10, 0.10]
)

# Regions — based on countries in real data
regions = np.random.choice(
    ['UK', 'Germany', 'France', 'Netherlands', 'EIRE', 'Other Europe'],
    size=n,
    p=[0.60, 0.12, 0.10, 0.06, 0.07, 0.05]
)

# Introduce ~5% missing country
missing_region_idx = np.random.choice(n, size=int(n * 0.05), replace=False)
regions_array = np.array(regions, dtype=object)
regions_array[missing_region_idx] = np.nan

# Acquisition date — before first transaction (synthetic)
base_date = datetime(2009, 1, 1)
acq_dates = [
    (base_date + timedelta(days=int(np.random.uniform(0, 365)))).strftime('%Y-%m-%d')
    for _ in range(n)
]

customers_df = pd.DataFrame({
    'customer_id': known_customers,
    'segment': segments,
    'clv': clv_array,
    'acquisition_channel': channels,
    'region': regions_array,
    'acquisition_date': acq_dates
})

# Introduce ~2% duplicate rows (system sync error)
dup_idx = np.random.choice(len(customers_df), size=int(n * 0.02), replace=False)
duplicates = customers_df.iloc[dup_idx].copy()
customers_df = pd.concat([customers_df, duplicates], ignore_index=True)

# Shuffle so duplicates aren't all at the bottom
customers_df = customers_df.sample(frac=1, random_state=42).reset_index(drop=True)

customers_df.to_csv('data/raw/customers_raw.csv', index=False)

print(f'customers_raw.csv saved — {len(customers_df):,} rows')
print(f'Nulls: {customers_df.isnull().sum().to_dict()}')
print(customers_df.head(5))

customers_raw.csv saved — 6,060 rows
Nulls: {'customer_id': 0, 'segment': 0, 'clv': 490, 'acquisition_channel': 0, 'region': 300, 'acquisition_date': 0}
   customer_id    segment       clv acquisition_channel        region  \
0        17976     Retail       NaN            Referral   Netherlands   
1        16608  Wholesale   1016.96              Online  Other Europe   
2        14596  Corporate  14379.43              Online        France   
3        12499     Retail    819.53          Trade Show            UK   
4        16624     Retail    116.46        Direct Sales            UK   

  acquisition_date  
0       2009-04-21  
1       2009-11-16  
2       2009-09-02  
3       2009-06-22  
4       2009-05-28  


---
## 4. Generate Service Tickets Table (Synthetic)

Each ticket represents a customer service interaction — a complaint, query, or return request. Tickets are linked to real customer IDs.

**Realistic patterns deliberately built in:**
- Corporate customers raise fewer but higher-severity tickets
- SLA breaches cluster at specific locations (Location 3 and Location 7 are problematic)
- Billing complaints have the worst resolution times
- CSAT scores drop when resolution time exceeds 5 days
- Seasonal spike in complaints in November-December (peak trading)

**Deliberate messiness introduced:**
- Inconsistent issue category naming
- Missing CSAT scores (~18% — customers who did not respond)
- Missing resolution times for abandoned tickets (~6%)
- Open tickets with no follow-up date

In [8]:
n_tickets = 15000

# Sample customer IDs for tickets (customers with issues)
ticket_customers = np.random.choice(known_customers, size=n_tickets, replace=True)

# Locations — 8 service locations, 3 and 7 are problematic
locations = np.random.choice(
    ['LOC_01', 'LOC_02', 'LOC_03', 'LOC_04', 'LOC_05', 'LOC_06', 'LOC_07', 'LOC_08'],
    size=n_tickets,
    p=[0.15, 0.12, 0.18, 0.10, 0.10, 0.08, 0.17, 0.10]  # LOC_03 and LOC_07 carry more volume
)

# Issue categories with deliberate naming inconsistency (messiness)
issue_pool = [
    'Billing Issue', 'billing issue', 'Billing',          # same category, 3 names
    'Delivery Delay', 'delivery delay', 'Late Delivery',  # same category, 3 names
    'Product Defect', 'Defective Product',                # same category, 2 names
    'Refund Request', 'refund',                           # same category, 2 names
    'Account Access', 'Login Issue',                      # same category, 2 names
    'General Enquiry'
]

issue_weights = [
    0.10, 0.05, 0.05,   # Billing
    0.10, 0.05, 0.05,   # Delivery
    0.08, 0.05,         # Defect
    0.10, 0.07,         # Refund
    0.07, 0.08,         # Account
    0.15                # General
]

issue_categories = np.random.choice(issue_pool, size=n_tickets, p=issue_weights)

# Priority levels
priorities = np.random.choice(['Low', 'Medium', 'High', 'Critical'], size=n_tickets, p=[0.30, 0.40, 0.20, 0.10])

# Ticket dates — spanning 2010-2011, with Nov-Dec spike
def generate_ticket_date():
    month_weights = [5,4,5,5,5,5,5,5,5,6,10,10]  # Nov and Dec heavier
    month_weights = [w/sum(month_weights) for w in month_weights]
    year = np.random.choice([2010, 2011])
    month = np.random.choice(range(1, 13), p=month_weights)
    day = np.random.randint(1, 28)
    return datetime(year, month, day)

ticket_dates = [generate_ticket_date() for _ in range(n_tickets)]

# Resolution time in days — billing worst, general best
# LOC_03 and LOC_07 have longer resolution times
def get_resolution_days(issue, location):
    base_map = {
        'billing': np.random.uniform(4, 10),
        'delivery': np.random.uniform(2, 7),
        'defect': np.random.uniform(3, 8),
        'refund': np.random.uniform(3, 9),
        'account': np.random.uniform(1, 4),
        'general': np.random.uniform(0.5, 3)
    }
    issue_lower = issue.lower()
    for key in base_map:
        if key in issue_lower:
            days = base_map[key]
            break
    else:
        days = np.random.uniform(1, 5)

    # LOC_03 and LOC_07 add delay
    if location in ['LOC_03', 'LOC_07']:
        days *= np.random.uniform(1.3, 1.8)

    return round(days, 1)

resolution_days = [get_resolution_days(issue, loc) for issue, loc in zip(issue_categories, locations)]

# Introduce ~6% missing resolution time (abandoned tickets)
missing_res_idx = np.random.choice(n_tickets, size=int(n_tickets * 0.06), replace=False)
resolution_array = np.array(resolution_days, dtype=object)
resolution_array[missing_res_idx] = np.nan

# SLA target is 2 days — breach if resolution > 2
sla_breached = [
    None if r is np.nan else (1 if float(r) > 2 else 0)
    for r in resolution_array
]

# CSAT score — 1 to 5, lower when resolution time is longer
def get_csat(resolution):
    if resolution is np.nan or resolution is None:
        return np.nan
    r = float(resolution)
    if r <= 1:
        return np.random.choice([4, 5], p=[0.35, 0.65])
    elif r <= 2:
        return np.random.choice([3, 4, 5], p=[0.20, 0.50, 0.30])
    elif r <= 5:
        return np.random.choice([2, 3, 4], p=[0.30, 0.50, 0.20])
    else:
        return np.random.choice([1, 2, 3], p=[0.40, 0.40, 0.20])

csat_scores = [get_csat(r) for r in resolution_array]

# Introduce ~18% missing CSAT (customers who did not respond)
missing_csat_idx = np.random.choice(n_tickets, size=int(n_tickets * 0.18), replace=False)
csat_array = np.array(csat_scores, dtype=object)
csat_array[missing_csat_idx] = np.nan

# Ticket status
statuses = np.random.choice(['Resolved', 'Open', 'Escalated'], size=n_tickets, p=[0.75, 0.15, 0.10])

# Ticket IDs
ticket_ids = [f'TKT-{str(i).zfill(5)}' for i in range(1, n_tickets + 1)]

tickets_df = pd.DataFrame({
    'ticket_id': ticket_ids,
    'customer_id': ticket_customers,
    'location_id': locations,
    'ticket_date': [d.strftime('%Y-%m-%d') for d in ticket_dates],
    'issue_category': issue_categories,
    'priority': priorities,
    'resolution_days': resolution_array,
    'sla_breached': sla_breached,
    'csat_score': csat_array,
    'status': statuses
})

tickets_df.to_csv('data/raw/tickets_raw.csv', index=False)

print(f'tickets_raw.csv saved — {len(tickets_df):,} rows')
print(f'Nulls: {tickets_df.isnull().sum().to_dict()}')
print(tickets_df.head(5))

tickets_raw.csv saved — 15,000 rows
Nulls: {'ticket_id': 0, 'customer_id': 0, 'location_id': 0, 'ticket_date': 0, 'issue_category': 0, 'priority': 0, 'resolution_days': 900, 'sla_breached': 900, 'csat_score': 3464, 'status': 0}
   ticket_id  customer_id location_id ticket_date   issue_category  priority  \
0  TKT-00001        13641      LOC_02  2011-07-12   Delivery Delay       Low   
1  TKT-00002        15780      LOC_03  2011-07-04   Refund Request       Low   
2  TKT-00003        15481      LOC_01  2011-05-16   Refund Request    Medium   
3  TKT-00004        17274      LOC_01  2011-02-14  General Enquiry    Medium   
4  TKT-00005        14196      LOC_08  2010-06-07           refund  Critical   

  resolution_days  sla_breached csat_score     status  
0             6.2           1.0          2   Resolved  
1             8.2           1.0          3       Open  
2             3.7           1.0          4  Escalated  
3             NaN           NaN        NaN   Resolved  
4          

---
## 5. Generate Operations Table (Synthetic)

One row per location per quarter — representing operational metrics reported at each service location. Linked to tickets via `location_id`.

**Realistic patterns deliberately built in:**
- LOC_03 and LOC_07 consistently miss SLA targets
- LOC_03 has understaffing issues
- Q4 shows inventory stress across all locations (peak trading)

**Deliberate messiness introduced:**
- Missing SLA achievement for two locations in one quarter
- Missing staff count for one location
- Inconsistent location naming in one quarter (LOC_03 vs Location_03)

In [9]:
location_ids = ['LOC_01', 'LOC_02', 'LOC_03', 'LOC_04', 'LOC_05', 'LOC_06', 'LOC_07', 'LOC_08']
quarters = ['2010-Q1', '2010-Q2', '2010-Q3', '2010-Q4', '2011-Q1', '2011-Q2', '2011-Q3', '2011-Q4']

ops_rows = []

for quarter in quarters:
    for loc in location_ids:
        is_q4 = 'Q4' in quarter
        is_problematic = loc in ['LOC_03', 'LOC_07']

        # Staff count
        if loc == 'LOC_03':
            staff = np.random.randint(3, 6)   # understaffed
        else:
            staff = np.random.randint(8, 20)

        # SLA target (days) — all locations target 2 days
        sla_target = 2.0

        # SLA achievement rate — % of tickets resolved within target
        if is_problematic:
            sla_achievement = round(np.random.uniform(0.42, 0.62), 2)  # consistently poor
        else:
            sla_achievement = round(np.random.uniform(0.72, 0.95), 2)  # mostly good

        # Inventory level — drops in Q4
        if is_q4:
            inventory = round(np.random.uniform(0.45, 0.70), 2)   # stressed
        else:
            inventory = round(np.random.uniform(0.70, 0.95), 2)

        # Monthly ticket volume handled
        if is_problematic:
            ticket_volume = np.random.randint(180, 320)
        else:
            ticket_volume = np.random.randint(80, 180)

        # Region
        region_map = {
            'LOC_01': 'UK', 'LOC_02': 'UK', 'LOC_03': 'Germany',
            'LOC_04': 'France', 'LOC_05': 'Netherlands',
            'LOC_06': 'EIRE', 'LOC_07': 'Germany', 'LOC_08': 'France'
        }

        ops_rows.append({
            'location_id': loc,
            'quarter': quarter,
            'region': region_map[loc],
            'staff_count': staff,
            'sla_target_days': sla_target,
            'sla_achievement_rate': sla_achievement,
            'inventory_level': inventory,
            'ticket_volume_handled': ticket_volume
        })

ops_df = pd.DataFrame(ops_rows)

# Introduce inconsistent naming for LOC_03 in 2010-Q2 (data entry error)
ops_df.loc[
    (ops_df['location_id'] == 'LOC_03') & (ops_df['quarter'] == '2010-Q2'),
    'location_id'
] = 'Location_03'

# Introduce missing SLA achievement for LOC_07 in 2010-Q1 and LOC_05 in 2011-Q3
ops_df.loc[
    (ops_df['location_id'] == 'LOC_07') & (ops_df['quarter'] == '2010-Q1'),
    'sla_achievement_rate'
] = np.nan

ops_df.loc[
    (ops_df['location_id'] == 'LOC_05') & (ops_df['quarter'] == '2011-Q3'),
    'sla_achievement_rate'
] = np.nan

# Introduce missing staff count for LOC_08 across all quarters
ops_df.loc[ops_df['location_id'] == 'LOC_08', 'staff_count'] = np.nan

ops_df.to_csv('data/raw/operations_raw.csv', index=False)

print(f'operations_raw.csv saved — {len(ops_df):,} rows')
print(f'Nulls: {ops_df.isnull().sum().to_dict()}')
print(ops_df.head(10))

operations_raw.csv saved — 64 rows
Nulls: {'location_id': 0, 'quarter': 0, 'region': 0, 'staff_count': 8, 'sla_target_days': 0, 'sla_achievement_rate': 2, 'inventory_level': 0, 'ticket_volume_handled': 0}
  location_id  quarter       region  staff_count  sla_target_days  \
0      LOC_01  2010-Q1           UK          9.0              2.0   
1      LOC_02  2010-Q1           UK         12.0              2.0   
2      LOC_03  2010-Q1      Germany          3.0              2.0   
3      LOC_04  2010-Q1       France          8.0              2.0   
4      LOC_05  2010-Q1  Netherlands          8.0              2.0   
5      LOC_06  2010-Q1         EIRE         14.0              2.0   
6      LOC_07  2010-Q1      Germany         13.0              2.0   
7      LOC_08  2010-Q1       France          NaN              2.0   
8      LOC_01  2010-Q2           UK         10.0              2.0   
9      LOC_02  2010-Q2           UK         12.0              2.0   

   sla_achievement_rate  inventory_

---
## 6. Final Summary — Data Generation Complete

In [2]:
import os

print('=' * 55)
print('  DATA GENERATION COMPLETE')
print('=' * 55)
print()

files = ['transactions_raw.csv', 'customers_raw.csv', 
         'tickets_raw.csv', 'operations_raw.csv']

for fname in files:
    path = f'data/raw/{fname}'
    size_kb = os.path.getsize(path) / 1024
    print(f'  data/raw/{fname}')
    print(f'    Size: {size_kb:.1f} KB')
    print()

print('All four raw tables saved.')


  DATA GENERATION COMPLETE

  data/raw/transactions_raw.csv
    Size: 100156.6 KB

  data/raw/customers_raw.csv
    Size: 273.8 KB

  data/raw/tickets_raw.csv
    Size: 1052.1 KB

  data/raw/operations_raw.csv
    Size: 2.8 KB

All four raw tables saved.
